# Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

In [ ]:
import torchvision.transforms as transforms

In [ ]:
import warnings
import matplotlib.pyplot as plt

from tqdm import tqdm
from google.colab import drive
from gensim.models.keyedvectors import KeyedVectors # This line doesn't load the trained model

## set and check something

In [ ]:
torch.manual_seed(1)

In [ ]:
warnings.filterwarnings(action="ignore", category=UserWarning, module="gensim")

In [ ]:
if torch.cuda.is_available():
  device = torch.device("cuda:0")
  print("GPU")
else:
  device = torch.device("cpu")
  print("CPU")

GPU


# **Define classes and functions**

## MyDataset class

In [ ]:
class MyDataset(Dataset):

    def __init__(self, data_csv_file, data_dir, label_dir, transform=None):
      self.transform = transform

      self.data_dir = data_dir
      data_dircsv_file = os.path.join(self.data_dir, csv_file)

      self.my_data = pd.read_csv(data_dircsv_file)

      self.len = self.my_data.shape[0]
        
    def __getitem__(self, index):
      # our report text is denoted as x
      x = self.my_data.iloc[index, 0]

      # our label is denoted as y
      y = self.my_data.iloc[index, 1]

      sample = x, y

      if self.transform:
        sample = self.transform(sample)

      return sample
    
    def __len__(self):
        return self.len

### Load data from google drive

In [ ]:
drive.mount("/content/gdrive", force_remount=True)
!ls "/content/gdrive/MyDrive/"

#### Load the pre-trainded word2vec model
- Our pre-trained model is: GoogleNews-vectors-negative300.bin.gz

- You can find this pre-trained model at: https://code.google.com/archive/p/word2vec/

In [ ]:
# this is how you load the model
directory = "gdrive/MyDrive/Colab Notebooks/00- data/GoogleNews-vectors-negative300.bin"
model = KeyedVectors.load_word2vec_format(directory, binary=True)

#### Limit model size while loading
Your system is freezing because of the large size of model. Try using system with more memory or you can limit the size of model you are loading.

In [ ]:
# model = KeyedVectors.load_word2vec_format(path_to_model, binary=True, limit=20000)

## **CNN class**

In [ ]:
class MyCNN(nn.Module):
    
    # Constructor
    def __init__(self, sequence_length, filter_sizes, out_1=256):
        
        super(CNN, self).__init__()
        
        self.cnns = nn.ModuleList()
        self.maxpools = nn.ModuleList()
        
        self.cnn_list = []
        self.maxpool_list = []
                
        for i, filter_size in enumerate(filter_sizes):
            cnn1 = nn.Conv2d(in_channels=1, out_channels=256, kernel_size=(filter_size, 6), stride=1, padding=0)
            maxpool1 = nn.MaxPool2d(kernel_size=(sequence_length - filter_size + 1, 1), stride=1, padding=0)
            
            self.cnns.append(cnn1)
            self.maxpools.append(maxpool1)
    
    # Prediction
    def forward(self, x):
        
        self.pooled_outputs = []
        
        for i, filter_size in enumerate(filter_sizes):
            x1 = self.cnns[i](x)
            print(x1.shape)
            x1 = torch.relu(x1)
            print(x1.shape)
            x1 = self.maxpools[i](x1)
            print(x1.shape)
            self.pooled_outputs.append(x1)
        return self.pooled_outputs

## train function

In [ ]:
def train_model(train_data_set, train_data_loader, model, criterion, optimizer, epochs=5):
  loss_total = []

  for epoch in tqdm(range(epochs)):
    cost = 0
    for features, labels in train_data_loader:
      optimizer.zero_grad()
      yhat = model(features)
      loss = criterion(yhat, label)
      loss.backward()
      optimizer.step()
      loss_total.append(loss.item())
    print("\nloss: ", loss, "\n")
  return loss_total

## test function

In [ ]:
def test_model(test_data_set, test_data_loader, model, criterion, optimizer, epochs=5):
  model.eval()
  num_correct = 0
  num_total = 0

  with torch.no_grad():
    for features, labels in test_data_loader:
      yhat = model(features)
      for index, tensor_value in enumerate(yhat):
        num_total += 1
        if torch.argmax(tensor_value) == labels[index]:
          num_correct += 1

  accuracy = num_correct / num_total
  print(f"Accuracy: {accuracy}")
  return accuracy

# **Create Objects**

## Data object

dataset

In [ ]:
data_set = MyDataset()

train_data_set = DataLoader(dataset=data_set, batch_size=10)
test_data_set = DataLoader(dataset=data_set, batch_size=10)

dataloader

In [ ]:
train_data_loader = DataLoader(dataset=train_data_set, batch_size=10, shuffle=True)
test_data_loader = DataLoader(dataset=test_data_set, batch_size=10, shuffle=True)

In [ ]:
train_data_loader.to(device)
test_data_loader.to(device)

## model object

In [ ]:
filter_sizes = [3, 4]
sequence_length = 10

In [ ]:
model = MyCNN(sequence_length, filter_sizes)
model.to(device)

In [ ]:
model

In [ ]:
yhat = model(data_set.x)

In [ ]:
print(len(yhat))
count = 0
for j in range(len(yhat)):
    print(yhat[j].shape)
    count += yhat[j].shape[1]
print(count)

In [ ]:
torch.cat(yhat[0], 0)

In [ ]:
v = torch.cat(yhat, 3)

## criterion and optimizer objects

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.params(), lr=0.01)

## train the model

In [ ]:
n_epochs = 10
cost_list = []
accuracy_list = []
n_test = len(validation_dataset)
cost = 0

In [ ]:
train_model(train_data_set, train_data_loader, model, criterion, optimizer, epochs=5)

## test the model

In [ ]:
accuracy = test_model(train_data_set, train_data_loader, model, criterion, optimizer, epochs=5)